### Célula 1 — Lendo o arquivo bruto

In [0]:
import pyspark.sql.functions as F

# Lendo o arquivo bruto — sem nenhuma transformação
RAW_FILE_PATH = "/Volumes/workspace/default/raw/customer_support_tickets - original.csv"

df_raw = (spark.read
    .option("header", "true")                                        # primeira linha é cabeçalho
    .option("inferSchema", "false")                                  # tudo como string
    .option("multiLine", "true")                                     # descrições com quebra de linha
    .option("escape", '"')                                           # aspas escapadas
    .csv(RAW_FILE_PATH)
)

df_raw = df_raw.toDF(*[c.replace(" ", "_") for c in df_raw.columns]) # padroniza nomes

print(f"✅ Arquivo lido com sucesso!")
print(f"Total de linhas:   {df_raw.count()}")
print(f"Total de colunas:  {len(df_raw.columns)}")

✅ Arquivo lido com sucesso!
Total de linhas:   8469
Total de colunas:  17


### Célula 2 — Schema e tipos originais

In [0]:
# Visualizando o schema original — tudo como string (dado bruto)
print("=" * 50)
print("SCHEMA DO DADO BRUTO")
print("=" * 50)
df_raw.printSchema()

# Amostra das primeiras linhas
print("\n")
print("=" * 50)
print("AMOSTRA — 5 PRIMEIRAS LINHAS")
print("=" * 50)
df_raw.show(5, truncate=40)

SCHEMA DO DADO BRUTO
root
 |-- Ticket_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Customer_Email: string (nullable = true)
 |-- Customer_Age: string (nullable = true)
 |-- Customer_Gender: string (nullable = true)
 |-- Product_Purchased: string (nullable = true)
 |-- Date_of_Purchase: string (nullable = true)
 |-- Ticket_Type: string (nullable = true)
 |-- Ticket_Subject: string (nullable = true)
 |-- Ticket_Description: string (nullable = true)
 |-- Ticket_Status: string (nullable = true)
 |-- Resolution: string (nullable = true)
 |-- Ticket_Priority: string (nullable = true)
 |-- Ticket_Channel: string (nullable = true)
 |-- First_Response_Time: string (nullable = true)
 |-- Time_to_Resolution: string (nullable = true)
 |-- Customer_Satisfaction_Rating: string (nullable = true)



AMOSTRA — 5 PRIMEIRAS LINHAS
+---------+-------------------+--------------------------+------------+---------------+-----------------+----------------+---------------+----

### Célula 3 — Análise de nulos

In [0]:
import builtins

# Contagem de nulos por coluna
print("=" * 60)
print("ANÁLISE DE NULOS POR COLUNA")
print("=" * 60)

total = df_raw.count()

nulos_data = []
for coluna in df_raw.columns:
    nulos = df_raw.filter(
        F.col(coluna).isNull() |                                     # nulo real
        (F.trim(F.col(coluna)) == "")                               # string vazia
    ).count()
    pct = builtins.round(nulos / total * 100, 1)
    nulos_data.append((coluna, nulos, pct))

# Exibe resultado
print(f"{'Coluna':<35} {'Nulos':>8} {'%':>8}")
print("-" * 55)
for coluna, nulos, pct in sorted(nulos_data, key=lambda x: -x[1]):
    status = "🔴" if pct > 50 else "🟡" if pct > 0 else "✅"
    print(f"{status} {coluna:<33} {nulos:>8} {pct:>7}%")

print("-" * 55)
print(f"{'Total de registros':<35} {total:>8}")

ANÁLISE DE NULOS POR COLUNA
Coluna                                 Nulos        %
-------------------------------------------------------
🔴 Resolution                            5700    67.3%
🔴 Time_to_Resolution                    5700    67.3%
🔴 Customer_Satisfaction_Rating          5700    67.3%
🟡 First_Response_Time                   2819    33.3%
✅ Ticket_ID                                0     0.0%
✅ Customer_Name                            0     0.0%
✅ Customer_Email                           0     0.0%
✅ Customer_Age                             0     0.0%
✅ Customer_Gender                          0     0.0%
✅ Product_Purchased                        0     0.0%
✅ Date_of_Purchase                         0     0.0%
✅ Ticket_Type                              0     0.0%
✅ Ticket_Subject                           0     0.0%
✅ Ticket_Description                       0     0.0%
✅ Ticket_Status                            0     0.0%
✅ Ticket_Priority                          0     0.0

### Célula 4 — Análise de duplicatas

In [0]:
import builtins

print("=" * 60)
print("ANÁLISE DE DUPLICATAS")
print("=" * 60)

# Duplicatas por Ticket_ID
total         = df_raw.count()                                       # total de registros
unicos        = df_raw.select("Ticket_ID").distinct().count()        # IDs únicos
duplicatas    = total - unicos                                        # diferença

print(f"Total de registros:      {total:>8,}")
print(f"Ticket_IDs únicos:       {unicos:>8,}")
print(f"Duplicatas encontradas:  {duplicatas:>8,}")
print(f"Status: {'✅ Sem duplicatas' if duplicatas == 0 else '🔴 Duplicatas encontradas!'}")

print()
print("=" * 60)
print("ANÁLISE DE UNICIDADE POR COLUNA")
print("=" * 60)

# Unicidade das colunas categóricas
colunas_cat = [
    "Ticket_Type", "Ticket_Status", "Ticket_Priority",
    "Ticket_Channel", "Ticket_Subject", "Customer_Gender",
    "Product_Purchased"
]

print(f"{'Coluna':<35} {'Únicos':>8}")
print("-" * 45)
for col in colunas_cat:
    unicos_col = df_raw.select(col).distinct().count()
    print(f"  {col:<33} {unicos_col:>8,}")

ANÁLISE DE DUPLICATAS
Total de registros:         8,469
Ticket_IDs únicos:          8,469
Duplicatas encontradas:         0
Status: ✅ Sem duplicatas

ANÁLISE DE UNICIDADE POR COLUNA
Coluna                                Únicos
---------------------------------------------
  Ticket_Type                              5
  Ticket_Status                            3
  Ticket_Priority                          4
  Ticket_Channel                           4
  Ticket_Subject                          16
  Customer_Gender                          3
  Product_Purchased                       42


### Célula 5 — Análise de distribuição e anomalias numéricas

In [0]:
import builtins

print("=" * 60)
print("ANÁLISE DE COLUNAS NUMÉRICAS — DADO BRUTO")
print("=" * 60)

# Customer_Age
df_age = df_raw.select(
    F.min("Customer_Age").alias("min"),
    F.max("Customer_Age").alias("max"),
    F.avg(F.col("Customer_Age").cast("double")).alias("media"),
    F.count("Customer_Age").alias("total")
).collect()[0]

print(f"\n📊 Customer_Age")
print(f"  Mínimo:  {df_age['min']}")
print(f"  Máximo:  {df_age['max']}")
print(f"  Média:   {builtins.round(df_age['media'], 1)}")

# Valores fora do range esperado
anomalias_age = df_raw.filter(
    (F.col("Customer_Age").cast("integer") < 18) |
    (F.col("Customer_Age").cast("integer") > 100)
).count()
print(f"  Anomalias (<18 ou >100): {anomalias_age} {'✅' if anomalias_age == 0 else '🔴'}")

# Customer_Satisfaction_Rating
df_sat = df_raw.filter(
    F.col("Customer_Satisfaction_Rating").isNotNull()
).select(
    F.min("Customer_Satisfaction_Rating").alias("min"),
    F.max("Customer_Satisfaction_Rating").alias("max"),
    F.avg(F.col("Customer_Satisfaction_Rating").cast("double")).alias("media")
).collect()[0]

print(f"\n📊 Customer_Satisfaction_Rating")
print(f"  Mínimo:  {df_sat['min']}")
print(f"  Máximo:  {df_sat['max']}")
print(f"  Média:   {builtins.round(df_sat['media'], 2)}")

anomalias_sat = df_raw.filter(
    F.col("Customer_Satisfaction_Rating").isNotNull() &
    ((F.col("Customer_Satisfaction_Rating").cast("double") < 1) |
     (F.col("Customer_Satisfaction_Rating").cast("double") > 5))
).count()
print(f"  Anomalias (fora de 1-5): {anomalias_sat} {'✅' if anomalias_sat == 0 else '🔴'}")

print()
print("=" * 60)
print("ANÁLISE DO TICKET_DESCRIPTION — PLACEHOLDER")
print("=" * 60)

placeholder = df_raw.filter(
    F.col("Ticket_Description").contains("{product_purchased}")
).count()
print(f"  Registros com placeholder: {placeholder:,} {'🔴 Requer limpeza' if placeholder > 0 else '✅'}")
print(f"  Percentual:                {builtins.round(placeholder/df_raw.count()*100,1)}%")

ANÁLISE DE COLUNAS NUMÉRICAS — DADO BRUTO

📊 Customer_Age
  Mínimo:  18
  Máximo:  70
  Média:   44.0
  Anomalias (<18 ou >100): 0 ✅

📊 Customer_Satisfaction_Rating
  Mínimo:  1.0
  Máximo:  5.0
  Média:   2.99
  Anomalias (fora de 1-5): 0 ✅

ANÁLISE DO TICKET_DESCRIPTION — PLACEHOLDER
  Registros com placeholder: 8,469 🔴 Requer limpeza
  Percentual:                100.0%


### Célula 6 — Análise de consistência entre colunas

In [0]:
import builtins

print("=" * 60)
print("ANÁLISE DE CONSISTÊNCIA ENTRE COLUNAS")
print("=" * 60)

# Tickets Closed sem Resolution
closed_sem_resolution = df_raw.filter(
    (F.col("Ticket_Status") == "Closed") &
    (F.col("Resolution").isNull())
).count()
print(f"\n🔍 Tickets 'Closed' sem Resolution:")
print(f"   {closed_sem_resolution} {'✅ Consistente' if closed_sem_resolution == 0 else '🔴 Inconsistente'}")

# Tickets com Satisfaction mas não Closed
sat_sem_closed = df_raw.filter(
    F.col("Customer_Satisfaction_Rating").isNotNull() &
    (F.col("Ticket_Status") != "Closed")
).count()
print(f"\n🔍 Tickets com Satisfaction mas não Closed:")
print(f"   {sat_sem_closed} {'✅ Consistente' if sat_sem_closed == 0 else '🔴 Inconsistente'}")

# Tickets Closed sem Satisfaction
closed_sem_sat = df_raw.filter(
    (F.col("Ticket_Status") == "Closed") &
    F.col("Customer_Satisfaction_Rating").isNull()
).count()
print(f"\n🔍 Tickets 'Closed' sem Satisfaction Rating:")
print(f"   {closed_sem_sat} {'✅ Consistente' if closed_sem_sat == 0 else '🟡 Atenção'}")

# Tickets com Resolution_Time mas sem First_Response
res_sem_response = df_raw.filter(
    F.col("Time_to_Resolution").isNotNull() &
    F.col("First_Response_Time").isNull()
).count()
print(f"\n🔍 Tickets com Resolution mas sem First Response:")
print(f"   {res_sem_response} {'✅ Consistente' if res_sem_response == 0 else '🔴 Inconsistente'}")

print()
print("=" * 60)
print("RESUMO DO DIAGNÓSTICO")
print("=" * 60)
print("""
  ✅ Sem duplicatas em Ticket_ID
  ✅ Customer_Age dentro do range esperado (18-70)
  ✅ Customer_Satisfaction_Rating dentro do range (1-5)
  ✅ Tickets Closed sempre têm Resolution
  ✅ Consistência entre Resolution e First_Response
  🔴 100% das Ticket_Description com placeholder {{product_purchased}}
  🔴 67.3% de nulos em Resolution, Time_to_Resolution e Satisfaction
  🟡 33.3% de nulos em First_Response_Time
  🟡 Tickets Closed sem Satisfaction Rating — verificar regra de negócio
""")
print("=" * 60)
print("✅ Data Profiling concluído — dado aprovado para ingestão!")
print("=" * 60)

ANÁLISE DE CONSISTÊNCIA ENTRE COLUNAS

🔍 Tickets 'Closed' sem Resolution:
   0 ✅ Consistente

🔍 Tickets com Satisfaction mas não Closed:
   0 ✅ Consistente

🔍 Tickets 'Closed' sem Satisfaction Rating:
   0 ✅ Consistente

🔍 Tickets com Resolution mas sem First Response:
   0 ✅ Consistente

RESUMO DO DIAGNÓSTICO

  ✅ Sem duplicatas em Ticket_ID
  ✅ Customer_Age dentro do range esperado (18-70)
  ✅ Customer_Satisfaction_Rating dentro do range (1-5)
  ✅ Tickets Closed sempre têm Resolution
  ✅ Consistência entre Resolution e First_Response
  🔴 100% das Ticket_Description com placeholder {{product_purchased}}
  🔴 67.3% de nulos em Resolution, Time_to_Resolution e Satisfaction
  🟡 33.3% de nulos em First_Response_Time
  🟡 Tickets Closed sem Satisfaction Rating — verificar regra de negócio

✅ Data Profiling concluído — dado aprovado para ingestão!


Duplicatas✅ Zero

Anomalias numéricas✅ Zero

Consistência entre colunas✅ 100%

Placeholder🔴 100% das descrições

Nulos críticos🔴 67.3% em 3 colunas

First Response nulos🟡 33.3%